In [35]:
import pandas as pd

df_english = pd.read_csv('./cleaned_songs_with_lyrics.csv')

In [36]:
df_english[['track_genre','id']].groupby(by=["track_genre"]).count()

,id
track_genre,
afrobeat,96
alt-rock,65
alternative,75
ambient,124
anime,73
...,...
spanish,13
synth-pop,517
techno,82


In [37]:
df_genre_master = pd.read_csv('./genres.csv')

In [38]:
to_remove = ['guitar', 'happy', 'malay', 'sleep']

df_final_genre_list = df_english[~df_english['track_genre'].isin(to_remove)]

In [44]:
import re
subgenre = pd.read_csv('./genres.csv',index_col='subgenre')['parent'].str.lower()

df_final_genre_list['track_genre'] = [re.sub('[-]',"",x) for x in df_final_genre_list['track_genre'].to_list()]
df_final_genre_list['parent_genre'] = df_final_genre_list['track_genre'].map(subgenre)

df_for_model = df_final_genre_list[['parent_genre','track_genre','id','lyrics_cleaned']]



C:\Users\conno\AppData\Local\Temp\ipykernel_2052\2224587514.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_final_genre_list['track_genre'] = [re.sub('[-]',"",x) for x in df_final_genre_list['track_genre'].to_list()]
C:\Users\conno\AppData\Local\Temp\ipykernel_2052\2224587514.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_final_genre_list['parent_genre'] = df_final_genre_list['track_genre'].map(subgenre)


In [45]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import ComplementNB
from sklearn.metrics import classification_report



# 3. Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    df_for_model['lyrics_cleaned'], 
    df_for_model['parent_genre'], 
    test_size=0.2, 
    random_state=42,
    stratify=df_for_model['parent_genre'] # Important: ensures class distribution is preserved
)

# 4. Vectorize text (TF-IDF is generally better than raw counts for lyrics)
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# 5. Train using ComplementNB
# ComplementNB is specifically designed for imbalanced datasets
model = ComplementNB()
model.fit(X_train_vec, y_train)

# 6. Evaluate
predictions = model.predict(X_test_vec)
print(classification_report(y_test, predictions))

              precision    recall  f1-score   support

 alternative       0.33      0.19      0.24       391
       anime       0.00      0.00      0.00        15
       blues       0.00      0.00      0.00        36
    children       0.37      0.40      0.38        83
   classical       0.29      0.05      0.09        74
      comedy       0.62      0.19      0.29        26
     country       0.36      0.68      0.47       284
      disney       0.33      0.05      0.08        21
  electronic       0.48      0.57      0.52      1069
        folk       0.00      0.00      0.00        26
      gospel       0.00      0.00      0.00         1
     hip-hop       0.00      0.00      0.00        19
        jazz       0.12      0.06      0.08        16
       latin       0.00      0.00      0.00        11
       metal       0.42      0.74      0.53       448
         pop       0.19      0.08      0.11       265
    r&b/soul       0.23      0.09      0.13       132
      reggae       0.39    

C:\Users\conno\AppData\Roaming\Python\Python313\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\conno\AppData\Roaming\Python\Python313\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\conno\AppData\Roaming\Python\Python313\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capital

In [46]:
df_preds = pd.DataFrame({
    'lyrics': X_test,
    'actual_parent': y_test,
    'predicted_genre': predictions
})

In [47]:
correct = df_preds[df_preds['actual_parent'] == df_preds['predicted_genre']]

In [48]:
df_for_model.to_csv('text.csv', index=False)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score

# Split the data into features (X) and target (y)
X = df_for_model['lyrics_cleaned']
y = df_for_model['parent_genre']

# Split into training and test sets (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Create a pipeline: Vectorizer -> Logistic Regression
# We use 'lbfgs' solver which is good for multiclass problems
model_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, stop_words='english')),
    ('clf', LogisticRegression(solver='lbfgs', max_iter=1000,class_weight='balanced'))
])

# Train the model
print("Training model...")
model_pipeline.fit(X_train, y_train)

# Make predictions
predictions = model_pipeline.predict(X_test)

# Evaluate
print(f"Accuracy: {accuracy_score(y_test, predictions):.2f}")
print("\nClassification Report:\n")
print(classification_report(y_test, predictions))

# Try and combine a sentiment analysis on this to see if it improves 

Training model...
Accuracy: 0.30

Classification Report:

              precision    recall  f1-score   support

 alternative       0.34      0.26      0.29       391
       anime       0.23      0.47      0.30        15
       blues       0.07      0.22      0.10        36
    children       0.25      0.59      0.36        83
   classical       0.14      0.30      0.19        74
      comedy       0.16      0.42      0.23        26
     country       0.46      0.55      0.50       284
      disney       0.10      0.33      0.16        21
  electronic       0.65      0.13      0.22      1069
        folk       0.08      0.38      0.14        26
      gospel       0.00      0.00      0.00         1
     hip-hop       0.13      0.53      0.21        19
        jazz       0.06      0.12      0.08        16
       latin       0.03      0.18      0.05        11
       metal       0.51      0.65      0.57       448
         pop       0.21      0.14      0.17       265
    r&b/soul       0.18

In [ ]:
# Im thinking

# We use the 97 predefined genres. 

# We do sentiment analysis to try and extract the emotions from the song

# We esentially make a matrix and cluster the genres together inter a more manageable amount of groups

# We since these clusters would have more similarity, we now ignore sentiment, and only use word based methods to try to do supervised 
# learning and define the specific genre

# Then we can analyse to see if specific emotional charge have enough variation between genres.csv

# We can also see if the spotigfy defined genres are better than out clustered ones, and see if we can use our own judgement to identify why sentiment has 
# showed these variations and if we can identify the parent genre.